[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prashantkul/ai-ml-interviews-study-guide/blob/main/notebooks/week1_confidence_intervals.ipynb)


# Week 1 — Confidence Intervals That Matter for ML Eval Metrics

**Goal:** Stop reporting raw accuracy numbers without uncertainty. After this notebook you can answer the interview question:

> *"You said your model is 95% accurate. How confident are you in that 95% number?"*

We will:
1. Simulate a held-out eval (LAD-style binary safety eval) with true accuracy `p = 0.95` and `n = 200`.
2. Compute three confidence intervals around the observed accuracy:
   - **Wald** (normal approximation) — the textbook one, which is *bad* near `p ≈ 1`.
   - **Wilson** — the one you should actually use for proportions.
   - **Bootstrap percentile** — the model-free sanity check.
3. Compare them visually.
4. See how CI width shrinks as `n` grows (the `1/√n` rule).
5. Lock in a flashcard + a 90-second interview answer.

Reference: Sam Miller, *"Adding Error Bars to Evals"* (Anthropic, 2024).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

np.random.seed(42)

## 1. Synthetic LAD-style eval data

We pretend our model has a *true* accuracy of `p = 0.95` on a held-out safety eval, and we score it on `n = 200` examples. The number of correct answers is `Binomial(n, p)`.

In the real interview answer, swap `p_true` for whatever your actual eval number was.

In [ ]:
p_true = 0.95
n = 200

correct = np.random.binomial(n=n, p=p_true)
p_hat = correct / n

print(f"True accuracy (unknown in real life): {p_true}")
print(f"Sample size:                         n = {n}")
print(f"Observed correct:                    {correct}")
print(f"Observed accuracy p_hat:             {p_hat:.4f}")

## 2. Wald (normal-approximation) CI

$$
\hat{p} \;\pm\; z_{1-\alpha/2}\,\sqrt{\frac{\hat{p}(1-\hat{p})}{n}}
$$

Why it is bad near `p = 0` or `p = 1`:
- It is symmetric around `p_hat`, so it can extend **above 1.0** or **below 0.0**, which is nonsense for a probability.
- The standard-error term `sqrt(p(1-p)/n)` collapses to ~0 as `p_hat -> 1`, so the CI becomes artificially tiny exactly when you should be *least* confident.
- Coverage drops below the nominal 95% — sometimes a lot.

In [ ]:
def wald_ci(p_hat, n, alpha=0.05):
    z = norm.ppf(1 - alpha / 2)
    se = np.sqrt(p_hat * (1 - p_hat) / n)
    return p_hat - z * se, p_hat + z * se

wald_lo, wald_hi = wald_ci(p_hat, n)
print(f"Wald 95% CI:   [{wald_lo:.4f}, {wald_hi:.4f}]   width = {wald_hi - wald_lo:.4f}")
if wald_hi > 1.0:
    print("  ^ note: upper bound is > 1.0, which is nonsense.")

## 3. Wilson score CI

$$
\frac{\hat{p} + \dfrac{z^{2}}{2n} \;\pm\; z\,\sqrt{\dfrac{\hat{p}(1-\hat{p})}{n} + \dfrac{z^{2}}{4n^{2}}}}{1 + \dfrac{z^{2}}{n}}
$$

Properties to remember in the interview:
- **Asymmetric** around `p_hat` — pulls toward 0.5, which is exactly what you want when `p_hat` is near 0 or 1.
- **Always inside `[0, 1]`** — never gives you nonsense bounds.
- **Much better small-sample coverage** than Wald.
- This is the default choice for binomial proportion CIs in modern stats packages.

In [ ]:
def wilson_ci(p_hat, n, alpha=0.05):
    z = norm.ppf(1 - alpha / 2)
    denom = 1 + z**2 / n
    center = (p_hat + z**2 / (2 * n)) / denom
    half = (z * np.sqrt(p_hat * (1 - p_hat) / n + z**2 / (4 * n**2))) / denom
    return center - half, center + half

wilson_lo, wilson_hi = wilson_ci(p_hat, n)
print(f"Wilson 95% CI: [{wilson_lo:.4f}, {wilson_hi:.4f}]   width = {wilson_hi - wilson_lo:.4f}")

## 4. Bootstrap percentile CI

We resample the original 0/1 outcomes with replacement, compute the accuracy on each resample, and take the 2.5th and 97.5th percentiles of the resampled accuracies.

This makes no parametric assumption — it just asks *"if I had drawn a different test set, how much would my accuracy number wobble?"*

In [ ]:
outcomes = np.concatenate([np.ones(correct), np.zeros(n - correct)])

B = 10_000
rng = np.random.default_rng(42)
boot_means = np.empty(B)
for b in range(B):
    sample = rng.choice(outcomes, size=n, replace=True)
    boot_means[b] = sample.mean()

boot_lo, boot_hi = np.percentile(boot_means, [2.5, 97.5])
print(f"Bootstrap 95% CI: [{boot_lo:.4f}, {boot_hi:.4f}]   width = {boot_hi - boot_lo:.4f}")

## 5. Compare the three CIs

In [ ]:
methods = ["Wald", "Wilson", "Bootstrap"]
lows = [wald_lo, wilson_lo, boot_lo]
highs = [wald_hi, wilson_hi, boot_hi]
centers = [p_hat, p_hat, p_hat]
y = np.arange(len(methods))

fig, ax = plt.subplots(figsize=(8, 3.5))
for i, (lo, hi, c) in enumerate(zip(lows, highs, centers)):
    ax.plot([lo, hi], [i, i], lw=3)
    ax.plot(c, i, "o", color="black", zorder=5)
    ax.text(hi + 0.002, i, f"  [{lo:.3f}, {hi:.3f}]", va="center")

ax.axvline(p_true, ls="--", color="gray", label=f"true p = {p_true}")
ax.set_yticks(y)
ax.set_yticklabels(methods)
ax.set_xlabel("accuracy")
ax.set_xlim(0.85, 1.02)
ax.set_title(f"95% CIs for accuracy  (p_hat = {p_hat:.3f}, n = {n})")
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

## 6. Sample-size sensitivity: how fast does the CI shrink?

The standard error of a proportion is `sqrt(p(1-p)/n)`, so the CI half-width scales like `1/sqrt(n)`. Doubling your eval set only makes the CI ~1.41× tighter; to *halve* the CI you need **4×** the data.

In [ ]:
ns = [50, 100, 200, 500, 1000, 5000]
widths = []
for n_i in ns:
    lo, hi = wilson_ci(p_hat, n_i)
    widths.append(hi - lo)

for n_i, w in zip(ns, widths):
    print(f"n = {n_i:5d}   Wilson CI width = {w:.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ns, widths, "o-", label="Wilson CI width")

ref = widths[0] * np.sqrt(ns[0]) / np.sqrt(np.array(ns))
ax.plot(ns, ref, "--", color="gray", label=r"$\propto 1/\sqrt{n}$")

ax.set_xscale("log")
ax.set_xlabel("sample size n (log)")
ax.set_ylabel("95% Wilson CI width")
ax.set_title(f"CI width vs n  (holding p_hat = {p_hat:.3f})")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Flashcard

> **My LAD result is 95.5% [0.917, 0.976] on n = 200.**
>
> The CI is this wide because **n is small relative to how close `p` is to 1** — when `p_hat` is near the boundary, you need a lot of samples before any single error meaningfully changes the rate. 9 errors out of 200 vs 10 errors out of 200 already moves accuracy by 0.5 points.
>
> **Wilson is preferred over Wald** because Wald is symmetric and assumes a normal approximation to the binomial — near `p = 1` it produces upper bounds above 1.0 and badly under-covers. Wilson is asymmetric (pulled toward 0.5), always stays in `[0, 1]`, and has much better small-sample coverage.
>
> **Bootstrap gives a similar answer here because** for an i.i.d. sample of Bernoulli outcomes the bootstrap distribution of the mean converges to the same thing the Wilson formula approximates — they are estimating the same uncertainty, just by different routes. Bootstrap is the right tool when the metric is more complicated than a plain proportion (F1, AUC, calibration error, etc.).

## 8. Interview rehearsal

**Q: "You report 95% accuracy on your safety eval. How confident are you in that number?"**

**90-second answer (use the actual numbers from this notebook):**

> The 95% number is a point estimate on `n = 200` held-out examples. The honest way to report it is `95.5% [0.917, 0.976]` using a Wilson 95% CI. I cross-checked with a 10,000-iteration percentile bootstrap and got essentially the same interval, which is what you'd expect for a plain proportion.
>
> I deliberately don't use the textbook Wald interval here — near `p = 1` it's symmetric and can give upper bounds above 1.0, and its coverage drops well below 95%. Wilson is asymmetric, stays in `[0, 1]`, and has much better small-sample coverage.
>
> The interval is roughly six points wide, and that width is dominated by sample size, not by the model. Because CI half-width scales like `1/sqrt(n)`, halving the interval would require about 800 examples instead of 200. So if a stakeholder needs to know whether the true accuracy is, say, 93% vs 96%, the right next step isn't a better model — it's a bigger eval set, or a paired test against a baseline so the noise cancels.